# 3.1.3 因子分解机：FM

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

单独掌握稀疏二阶交互：从曝光—点击数据、FM 数学恒等式，到 PyTorch 训练、概率推理、AUC 与 LogLoss。

## Setup

本 Notebook 的默认真实数据是 **GroupLens MovieLens latest-small：经典评分与邻域任务**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [Factorization Machines](https://www.csie.ntu.edu.tw/~b97053/paper/Rendle2010FM.pdf)

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("RECSYS_PROFILE", "smoke")
PROFILE = os.environ["RECSYS_PROFILE"]
from recsys_lab.data import (load_movielens, movielens_provenance, load_amazon_2023,
                             amazon_provenance, load_kuairand, kuairand_provenance)
DATASET_KEY = "movielens"
if DATASET_KEY == "movielens":
    real_ratings, real_movies = load_movielens()
    real_interactions = real_ratings
    REAL_DATASET = movielens_provenance(real_ratings)
elif DATASET_KEY == "amazon-2023":
    real_ratings = load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_provenance(real_ratings)
else:
    real_interactions, real_movies = load_kuairand()
    real_ratings = real_interactions
    REAL_DATASET = kuairand_provenance(real_interactions)
print({"profile": PROFILE, "root": str(ROOT), "real_dataset": REAL_DATASET})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

{'profile': 'smoke', 'root': '/workspace', 'real_dataset': {'dataset': 'MovieLens latest-small (GroupLens, generated 2018-09-26)', 'source': 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', 'license_file': '/workspace/data/ml-latest-small/README.txt', 'rows_used': 26732, 'users_used': 120, 'items_used': 600, 'time_min_utc': '1996-10-17T11:51:49+00:00', 'time_max_utc': '2018-09-13T21:38:16+00:00', 'positive_rule': 'like := observed rating >= 4.0; very_like := observed rating >= 4.5', 'randomly_fabricated_rows': 0}}


## 来源论文与阅读提示

**Rendle (2010), Factorization Machines** 的核心贡献是把矩阵分解的隐向量交互推广到任意稀疏特征，并通过代数恒等式在线性时间内计算全部二阶项。阅读时应特别看稀疏条件下“未直接观察过的特征组合”如何借各自隐向量共享统计。

### 公式怎么读（直觉版）

先把 one-hot 理解成一排开关：当前用户、物品、设备对应的开关为 1，其余为 0。FM 为每个开关配置一张小坐标卡；任意两个打开的开关都做一次点积。恒等式不是新模型，只是把“逐对计算”重新整理为“先求和再平方”，避免重复工作。相关向量、点积和概率知识见 **3.0**。

In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error, roc_auc_score, log_loss

SEED = 2026
torch.manual_seed(SEED)
from recsys_lab.data import load_movielens, leave_last_out, movielens_provenance

ratings, movies = load_movielens(max_users=48, max_items=360, min_user_events=12)
train_ratings, test_ratings = leave_last_out(ratings)
n_users, n_items = ratings.user_id.nunique(), ratings.item_id.nunique()

print({
    "rows": len(ratings), "users": ratings.user_id.nunique(), "items": ratings.item_id.nunique(),
    "train_rows": len(train_ratings), "test_rows": len(test_ratings),
    "sparsity": round(1 - len(train_ratings) / (n_users * n_items), 3),
    "source": movielens_provenance(ratings)["source"], "fabricated_rows": 0,
})
display(ratings[["userId", "movieId", "rating", "timestamp", "title", "genres"]].head(8))

{'rows': 10487, 'users': 48, 'items': 360, 'train_rows': 10439, 'test_rows': 48, 'sparsity': 0.396, 'source': 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', 'fabricated_rows': 0}


,userId,movieId,rating,timestamp,title,genres
0,140,318,4.0,942840891,"Shawshank Redemption, The (1994)",Crime|Drama
1,140,527,5.0,942840891,Schindler's List (1993),Drama|War
2,140,1221,3.0,942840891,"Godfather: Part II, The (1974)",Crime|Drama
3,140,50,3.0,942840991,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
4,140,595,5.0,942840991,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX
5,140,1198,4.0,942841170,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure
6,140,2028,5.0,942841170,Saving Private Ryan (1998),Action|Drama|War
7,140,1721,4.0,942841219,Titanic (1997),Drama|Romance


---

## 4. 因子分解机（FM）：为稀疏特征学习二阶交互

### 4.1 为什么 MF 不够？

CTR 排序不只有 `user_id` 和 `item_id`，还会有设备、小时、地域、品类、价格等上下文。直接为每一对特征做 one-hot 交叉会导致参数爆炸；普通 LR 又只能做加法，无法表达“某用户群在晚间更喜欢某品类”。

FM 为每个特征 $i$ 学一个隐向量 $v_i$，用内积共享所有二阶交叉统计：

$$
\hat y(x)=w_0+\sum_i w_ix_i+\sum_{i<j}\langle v_i,v_j\rangle x_ix_j
$$

朴素二阶项是 $O(n^2)$，利用恒等式可化为 $O(nk)$：

$$
\sum_{i<j}\langle v_i,v_j\rangle x_ix_j
=\frac12\sum_f\left[\left(\sum_i v_{if}x_i\right)^2-\sum_i v_{if}^2x_i^2\right]
$$

当特征只有 user one-hot 和 item one-hot 时，FM 的交叉项就退化为 MF；因此 FM 可以理解为 MF 对通用稀疏特征的扩展。

### 4.2 数据：从真实评分构造可解释的二分类排序任务

In [3]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ctr = ratings.sort_values("timestamp").tail(5000).copy().reset_index(drop=True)
# `click` 是任务别名；标签由真实评分 rating >= 4.0 确定，不使用随机采样。
ctr["click"] = ctr["like"].astype(int)
split = int(len(ctr) * .8)  # 严格按真实 timestamp 切分
categorical = ["user_id", "item_id", "genre_id", "hour", "decade_id"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_sparse = encoder.fit_transform(ctr.loc[:split-1, categorical])
test_sparse = encoder.transform(ctr.loc[split:, categorical])
scaler = StandardScaler()
train_numeric = scaler.fit_transform(ctr.loc[:split-1, ["item_popularity", "user_activity"]])
test_numeric = scaler.transform(ctr.loc[split:, ["item_popularity", "user_activity"]])
fm_train_x = np.c_[train_sparse, train_numeric].astype("float32")
fm_test_x = np.c_[test_sparse, test_numeric].astype("float32")
ctr_train_y = ctr.loc[:split-1, "click"].to_numpy()
ctr_test_y = ctr.loc[split:, "click"].to_numpy()
print({"train": fm_train_x.shape, "test": fm_test_x.shape, "train_positive_rate": round(ctr_train_y.mean(), 3)})

{'train': (4000, 437), 'test': (1000, 437), 'train_positive_rate': np.float64(0.474)}


### 4.3 训练：显式实现 FM 的线性时间交叉

In [4]:
class FactorizationMachine(torch.nn.Module):
    def __init__(self, n_features, factor_dim=10):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)
        self.factors = torch.nn.Parameter(torch.randn(n_features, factor_dim) * 0.03)

    def forward(self, x):
        linear_logit = self.linear(x).squeeze(1)
        summed = x @ self.factors
        squared_sum = summed.pow(2)
        sum_squared = x.pow(2) @ self.factors.pow(2)
        pairwise_logit = 0.5 * (squared_sum - sum_squared).sum(dim=1)
        return linear_logit + pairwise_logit

fm_model = FactorizationMachine(fm_train_x.shape[1], factor_dim=10)
fm_optimizer = torch.optim.Adam(fm_model.parameters(), lr=.025, weight_decay=1e-5)
x_train_tensor = torch.tensor(fm_train_x)
y_train_tensor = torch.tensor(ctr_train_y, dtype=torch.float32)

for epoch in range(100):
    fm_logit = fm_model(x_train_tensor)
    fm_loss = torch.nn.functional.binary_cross_entropy_with_logits(fm_logit, y_train_tensor)
    fm_optimizer.zero_grad(); fm_loss.backward(); fm_optimizer.step()

print({"FM final BCE": round(float(fm_loss.detach()), 4)})

{'FM final BCE': 0.0102}


### 4.4 推理与测试：输出概率、AUC 与 LogLoss

In [5]:
fm_model.eval()
with torch.no_grad():
    fm_probability = torch.sigmoid(fm_model(torch.tensor(fm_test_x))).numpy()
fm_auc = float(roc_auc_score(ctr_test_y, fm_probability))
fm_logloss = float(log_loss(ctr_test_y, fm_probability))
display(pd.DataFrame({"label": ctr_test_y[:8], "p_click": fm_probability[:8].round(4)}))
print({"FM AUC": round(fm_auc, 4), "FM LogLoss": round(fm_logloss, 4)})

,label,p_click
0,0,0.0028
1,1,0.2287
2,0,0.0021
3,0,0.1610
4,0,0.0002
5,0,0.0159
6,0,0.3017
7,0,0.0062


{'FM AUC': 0.5964, 'FM LogLoss': 2.6061}


### 4.5 结果讨论与边界

AUC 关注正样本是否排在负样本前，LogLoss 关注概率本身是否可信。FM 能通过隐向量共享低频交叉的统计，但所有交互仍是二阶且形式相同。

**优点**：适合超稀疏 one-hot；无需枚举交叉；参数能跨组合共享；计算复杂度线性。  
**缺点**：主要建模二阶；不理解行为顺序；所有 field 共用同一种内积。  
**常见升级**：FFM 为不同 field 学不同向量；DeepFM 共享 embedding，同时加入 DNN 学高阶非线性。

## Checks

确认概率合法、AUC 高于随机水平，并同时观察 LogLoss，避免只看排序不看校准。

In [6]:
assert np.all((fm_probability >= 0) & (fm_probability <= 1))
assert fm_auc > .55
assert np.isfinite(fm_logloss)
print('PASS：FM 训练、概率推理和 CTR 指标均有效。')

PASS：FM 训练、概率推理和 CTR 指标均有效。


In [7]:
result_dir = ROOT / "results" / "chapter_3_1"; result_dir.mkdir(parents=True, exist_ok=True)
payload = {"records": [{"algorithm":"FM", "task":"CTR", "metric":"AUC ↑", "value":fm_auc, "secondary_metric":"LogLoss ↓", "secondary_value":fm_logloss, "online_inference":"线性项 + 二阶隐向量交互", "source_notebook":"3_1_3_factorization_machine"}]}
(result_dir / "factorization_machine.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写出章节指标：factorization_machine.json")

已写出章节指标：factorization_machine.json


## Next Steps

加入 LR 基线；拆解不同 field 的交互；继续学习 FFM、DeepFM 与 DCN。